# Music Industry Knowledge Graph & Behavioral Pattern Analytics

This notebook demonstrates:
1. **Audio Sources Gathering** (Free, EU-compliant audio via Jamendo & Archive.org)
2. **Music Industry Knowledge Graph Construction** (Artists, Record Labels, Agencies, Studios, Producers)
3. **Behavioral Pattern Analytics**:
   - **Power Brokers & Gatekeeper Centrality** (PageRank & Betweenness)
   - **Creative Ecosystems** (Community Detection & Genre Cliques)
   - **Label Mobility & Churn** (Loyalty vs. Label Hopping)
   - **Studio & Producer Reliance Index (SPRI)**
   - **Agency Collaboration Dynamics** (Walled Gardens vs. Open Ecosystems)
4. **Interactive Network Graph Visualization** with Pyvis.

In [ ]:
import sys
from pathlib import Path

# Ensure src directory is in Python path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import pandas as pd
from IPython.display import display, HTML

from agies.audio.manager import AudioSourcesManager
from agies.audio.models import AudioFilter
from agies.graph.builder import MusicIndustryGraph
from agies.graph.extractors.synthetic_extractor import SyntheticIndustryExtractor
from agies.analytics.patterns import MusicIndustryAnalytics
from agies.visualization.interactive import render_interactive_graph

print("✓ AGIES modules loaded successfully.")

## 1. Audio Sources Exploration (Jamendo & Archive.org)

Query audio tracks, view license metadata, and inspect streaming/download URLs.

In [ ]:
audio_manager = AudioSourcesManager()

# Query tracks from Jamendo
tracks = audio_manager.search(query="chill electronic", provider="jamendo", limit_per_provider=5)

tracks_data = []
for t in tracks:
    tracks_data.append({
        "ID": t.id,
        "Title": t.title,
        "Artist": t.artist,
        "Duration (s)": t.duration_seconds,
        "License": t.license.name,
        "Commercial OK": t.license.is_commercial_allowed,
        "Stream / Preview": t.stream_url,
    })

df_tracks = pd.DataFrame(tracks_data)
display(df_tracks)

## 2. Constructing the Music Industry Knowledge Graph

Ingest the interconnected ecosystem linking Artists, Record Labels, Agencies, Studios, and Producers.

In [ ]:
extractor = SyntheticIndustryExtractor()
entities, edges = extractor.extract()

industry_graph = MusicIndustryGraph()
industry_graph.ingest(entities, edges)

summary = industry_graph.summary()
print(f"Total Nodes: {summary['total_nodes']}")
print(f"Total Edges: {summary['total_edges']}")
print("\nNode Types:", summary["nodes_by_type"])
print("\nRelationships:", summary["edges_by_relationship"])

## 3. Power Broker & Gatekeeper Centrality

Identify dominant nodes acting as critical hubs or gatekeepers across the music industry.

In [ ]:
analytics = MusicIndustryAnalytics(industry_graph)
power_brokers = analytics.compute_power_brokers(top_k=8)

print("=== Top Power Brokers by PageRank (Network Prominence) ===")
display(pd.DataFrame(power_brokers["by_pagerank"]))

print("=== Top Gatekeepers by Betweenness Centrality (Information / Deal Flow Bridges) ===")
display(pd.DataFrame(power_brokers["by_betweenness"]))

## 4. Creative Ecosystems & Sub-Communities

Discover dense production cliques, label clusters, and collaborative micro-ecosystems.

In [ ]:
ecosystems = analytics.detect_creative_ecosystems()

comm_records = []
for c in ecosystems:
    comm_records.append({
        "Community #": c["community_id"],
        "Size": c["size"],
        "Dominant Genres": ", ".join(c["top_genres"]),
        "Prominent Members": ", ".join(c["prominent_members"][:4]),
    })

display(pd.DataFrame(comm_records))

## 5. Behavioral Pattern: Label Mobility vs. Single-Label Loyalty

Analyze artist churn rates, multi-label hopping behaviors, and historical label migrations.

In [ ]:
mobility = analytics.analyze_label_mobility()
print(f"Industry Label Migration Rate: {mobility['migration_rate_percentage']}%")
print(f"Single-Label Loyal Artists: {mobility['loyal_count']}")
print(f"Migrated / Multi-Label Artists: {mobility['migrated_count']}")

print("\n--- Artists with Label Migrations ---")
display(pd.DataFrame(mobility["migrated_artists"])[["artist_name", "total_labels_count", "current_labels", "past_labels"]])

## 6. Behavioral Pattern: Studio & Producer Reliance Index (SPRI)

Measure concentration of an artist's recording output within specific studios and producers.

In [ ]:
spri_results = analytics.compute_studio_reliance()
df_spri = pd.DataFrame(spri_results)
display(df_spri.head(10))

## 7. Behavioral Pattern: Agency Collaboration Dynamics

Analyze whether booking/management agencies operate as insular 'walled gardens' or cross-collaborative networks.

In [ ]:
agency_analysis = analytics.analyze_agency_collaboration_density()
print(f"Total Track Collaborations: {agency_analysis['total_collaborations']}")
print(f"Intra-Agency Collaborations: {agency_analysis['intra_agency_collaborations']}")
print(f"Inter-Agency Collaborations: {agency_analysis['inter_agency_collaborations']}")
print(f"Intra-Agency Ratio: {agency_analysis['intra_agency_ratio_percentage']}%")
print(f"Interpretation: {agency_analysis['behavior_interpretation']}")

## 8. Interactive Network Graph Visualization

Export and render the physics-simulated, color-coded interactive knowledge graph.

In [ ]:
html_path = render_interactive_graph(
    industry_graph,
    output_html_path="music_industry_network.html",
    heading="Music Industry Entity-Relationship & Behavioral Ecosystem",
)
print(f"Interactive Graph HTML generated at: {html_path.resolve()}")